In [3]:
!pwd

/data/users/jupyter-yos225/venvs/proflee/drug


In [5]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("./nsduh_analysis_outputs")

possible_files = [
    OUT_DIR / "classifier_comparison_full_corrected_7970_auc_pivot.csv",
    OUT_DIR / "all_classifier_comparison_full_corrected_7970_auc_pivot.csv",
    OUT_DIR / "all_models_full_corrected_7970_auc_pivot.csv",
    OUT_DIR / "learning_curve_full_corrected_7970_with_xgboost_auc_comparison.csv",
]

for p in possible_files:
    print(p, p.exists())

nsduh_analysis_outputs/classifier_comparison_full_corrected_7970_auc_pivot.csv False
nsduh_analysis_outputs/all_classifier_comparison_full_corrected_7970_auc_pivot.csv False
nsduh_analysis_outputs/all_models_full_corrected_7970_auc_pivot.csv False
nsduh_analysis_outputs/learning_curve_full_corrected_7970_with_xgboost_auc_comparison.csv True


In [6]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("./nsduh_analysis_outputs")

auc_pivot_path = OUT_DIR / "ml_results_full_corrected_7970_all_classifiers_auc_pivot.csv"

auc_pivot = pd.read_csv(auc_pivot_path)

print(auc_pivot.shape)
print(auc_pivot.columns.tolist())

auc_pivot

(7, 6)
['Model', 'GPT profile embedding', 'Raw + GPT profile embedding', 'Raw structured', 'GPT_minus_Raw', 'Combined_minus_Raw']


,Model,GPT profile embedding,Raw + GPT profile embedding,Raw structured,GPT_minus_Raw,Combined_minus_Raw
0,XGBoost,0.705182,0.705357,0.686337,0.018845,0.019019
1,Extra Trees,0.690974,0.690948,0.676303,0.014671,0.014645
2,HistGradientBoosting,0.686169,0.684434,0.671751,0.014417,0.012682
3,Random Forest,0.690550,0.690198,0.678975,0.011575,0.011222
4,Logistic Regression,0.697810,0.695972,0.687130,0.010680,0.008843
5,Ridge Classifier,0.686306,0.683998,0.686527,-0.000221,-0.002529
6,Linear SVM,0.685488,0.682650,0.686547,-0.001059,-0.003897


In [7]:
from scipy.stats import wilcoxon, binomtest
import numpy as np
import pandas as pd

# 필요한 column 확인
required_cols = [
    "Model",
    "GPT profile embedding",
    "Raw + GPT profile embedding",
    "Raw structured",
]

for col in required_cols:
    print(col, col in auc_pivot.columns)

# 만약 diff column이 없으면 새로 생성
if "GPT_minus_Raw" not in auc_pivot.columns:
    auc_pivot["GPT_minus_Raw"] = (
        auc_pivot["GPT profile embedding"] - auc_pivot["Raw structured"]
    )

if "Combined_minus_Raw" not in auc_pivot.columns:
    auc_pivot["Combined_minus_Raw"] = (
        auc_pivot["Raw + GPT profile embedding"] - auc_pivot["Raw structured"]
    )

wilcoxon_df = auc_pivot.dropna(
    subset=["GPT_minus_Raw", "Combined_minus_Raw"]
).copy()

wilcoxon_df = wilcoxon_df.sort_values("GPT_minus_Raw", ascending=False)

print("Number of classifiers:", len(wilcoxon_df))

display(
    wilcoxon_df[
        [
            "Model",
            "GPT profile embedding",
            "Raw + GPT profile embedding",
            "Raw structured",
            "GPT_minus_Raw",
            "Combined_minus_Raw",
        ]
    ]
)

Model True
GPT profile embedding True
Raw + GPT profile embedding True
Raw structured True
Number of classifiers: 7


,Model,GPT profile embedding,Raw + GPT profile embedding,Raw structured,GPT_minus_Raw,Combined_minus_Raw
0,XGBoost,0.705182,0.705357,0.686337,0.018845,0.019019
1,Extra Trees,0.690974,0.690948,0.676303,0.014671,0.014645
2,HistGradientBoosting,0.686169,0.684434,0.671751,0.014417,0.012682
3,Random Forest,0.690550,0.690198,0.678975,0.011575,0.011222
4,Logistic Regression,0.697810,0.695972,0.687130,0.010680,0.008843
5,Ridge Classifier,0.686306,0.683998,0.686527,-0.000221,-0.002529
6,Linear SVM,0.685488,0.682650,0.686547,-0.001059,-0.003897


In [8]:
gpt_diffs = wilcoxon_df["GPT_minus_Raw"].values
combined_diffs = wilcoxon_df["Combined_minus_Raw"].values

# One-sided Wilcoxon: median difference > 0
w_gpt = wilcoxon(
    gpt_diffs,
    alternative="greater",
    zero_method="wilcox"
)

w_combined = wilcoxon(
    combined_diffs,
    alternative="greater",
    zero_method="wilcox"
)

print("Wilcoxon signed-rank test: GPT profile embedding > Raw structured")
print("statistic:", w_gpt.statistic)
print("p-value:", w_gpt.pvalue)

print("\nWilcoxon signed-rank test: Raw + GPT profile embedding > Raw structured")
print("statistic:", w_combined.statistic)
print("p-value:", w_combined.pvalue)

Wilcoxon signed-rank test: GPT profile embedding > Raw structured
statistic: 25.0
p-value: 0.0390625

Wilcoxon signed-rank test: Raw + GPT profile embedding > Raw structured
statistic: 25.0
p-value: 0.0390625


In [9]:
gpt_diffs = wilcoxon_df["GPT_minus_Raw"].values
combined_diffs = wilcoxon_df["Combined_minus_Raw"].values

# One-sided Wilcoxon: median difference > 0
w_gpt = wilcoxon(
    gpt_diffs,
    alternative="greater",
    zero_method="wilcox"
)

w_combined = wilcoxon(
    combined_diffs,
    alternative="greater",
    zero_method="wilcox"
)

print("Wilcoxon signed-rank test: GPT profile embedding > Raw structured")
print("statistic:", w_gpt.statistic)
print("p-value:", w_gpt.pvalue)

print("\nWilcoxon signed-rank test: Raw + GPT profile embedding > Raw structured")
print("statistic:", w_combined.statistic)
print("p-value:", w_combined.pvalue)

Wilcoxon signed-rank test: GPT profile embedding > Raw structured
statistic: 25.0
p-value: 0.0390625

Wilcoxon signed-rank test: Raw + GPT profile embedding > Raw structured
statistic: 25.0
p-value: 0.0390625


In [ ]:
summary_test = pd.DataFrame([
    {
        "Comparison": "GPT profile embedding - Raw structured",
        "n_classifiers": len(gpt_diffs),
        "mean_diff": np.mean(gpt_diffs),
        "median_diff": np.median(gpt_diffs),
        "min_diff": np.min(gpt_diffs),
        "max_diff": np.max(gpt_diffs),
        "num_positive": int(np.sum(gpt_diffs > 0)),
        "num_negative": int(np.sum(gpt_diffs < 0)),
        "wilcoxon_statistic": w_gpt.statistic,
        "wilcoxon_p_one_sided": w_gpt.pvalue,
    },
    {
        "Comparison": "Raw + GPT profile embedding - Raw structured",
        "n_classifiers": len(combined_diffs),
        "mean_diff": np.mean(combined_diffs),
        "median_diff": np.median(combined_diffs),
        "min_diff": np.min(combined_diffs),
        "max_diff": np.max(combined_diffs),
        "num_positive": int(np.sum(combined_diffs > 0)),
        "num_negative": int(np.sum(combined_diffs < 0)),
        "wilcoxon_statistic": w_combined.statistic,
        "wilcoxon_p_one_sided": w_combined.pvalue,
    }
])

summary_test.to_csv(
    OUT_DIR / "wilcoxon_classifier_comparison_summary.csv",
    index=False
)

summary_test

,Comparison,n_classifiers,mean_diff,median_diff,min_diff,max_diff,num_positive,num_negative,wilcoxon_statistic,wilcoxon_p_one_sided
0,GPT profile embedding - Raw structured,7,0.009844,0.011575,-0.001059,0.018845,5,2,25.0,0.039062
1,Raw + GPT profile embedding - Raw structured,7,0.008569,0.011222,-0.003897,0.019019,5,2,25.0,0.039062


Across seven classifiers, GPT profile embeddings showed a positive AUC advantage over raw structured variables. The one-sided Wilcoxon signed-rank test was statistically significant, p = 0.039.

In [11]:
num_pos_gpt = int(np.sum(gpt_diffs > 0))
num_nonzero_gpt = int(np.sum(gpt_diffs != 0))

num_pos_combined = int(np.sum(combined_diffs > 0))
num_nonzero_combined = int(np.sum(combined_diffs != 0))

sign_gpt = binomtest(
    k=num_pos_gpt,
    n=num_nonzero_gpt,
    p=0.5,
    alternative="greater"
)

sign_combined = binomtest(
    k=num_pos_combined,
    n=num_nonzero_combined,
    p=0.5,
    alternative="greater"
)

sign_test_summary = pd.DataFrame([
    {
        "Comparison": "GPT profile embedding - Raw structured",
        "positive_differences": num_pos_gpt,
        "nonzero_differences": num_nonzero_gpt,
        "sign_test_p_one_sided": sign_gpt.pvalue,
    },
    {
        "Comparison": "Raw + GPT profile embedding - Raw structured",
        "positive_differences": num_pos_combined,
        "nonzero_differences": num_nonzero_combined,
        "sign_test_p_one_sided": sign_combined.pvalue,
    }
])

sign_test_summary.to_csv(
    OUT_DIR / "sign_test_classifier_comparison_summary.csv",
    index=False
)

sign_test_summary

,Comparison,positive_differences,nonzero_differences,sign_test_p_one_sided
0,GPT profile embedding - Raw structured,5,7,0.226562
1,Raw + GPT profile embedding - Raw structured,5,7,0.226562


In [1]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("./nsduh_analysis_outputs")

In [4]:
w = pd.read_csv(OUT_DIR / "transfer_drop_wilcoxon_results.csv").set_index("Comparison")

rows = w.loc[[
    "GPT vs Raw drop, mean of both directions",
    "GPT vs Raw drop, tree-based only (n=4)",
]]

table1 = pd.DataFrame({
    "Comparison": ["All five classifiers", "Tree-based only"],
    "n": rows["n"].astype(int).values,
    "Favoring GPT": [f"{int(k)}/{int(n)}" for k, n
                     in zip(rows["n_negative(favor GPT)"], rows["n"])],
    "Mean diff": rows["mean_diff"].round(4).values,
    "One-sided p": rows["wilcoxon_p_one_sided_less"].round(4).values,
})
table1.style.hide(axis="index")   # index 없이 깔끔하게 표시

Comparison,n,Favoring GPT,Mean diff,One-sided p
All five classifiers,5,4/5,-0.003400,0.312500
Tree-based only,4,4/4,-0.016400,0.062500


In [5]:
s = pd.read_csv(OUT_DIR / "cross_year_drop_summary_repeated_within_year_5seeds_by_model.csv")
s = s.sort_values("GPT_drop_minus_Raw_drop_mean")

table1 = pd.DataFrame({
    "Model": s["Model"],
    "Raw drop": s["Raw_drop_mean"].round(4),
    "GPT drop": s["GPT_drop_mean"].round(4),
    "GPT − Raw drop difference": s["GPT_drop_minus_Raw_drop_mean"].round(4),
})

table1.style.hide(axis="index")

Model,Raw drop,GPT drop,GPT − Raw drop difference
Random Forest,0.000900,-0.024300,-0.025100
Extra Trees,0.000800,-0.020800,-0.021600
XGBoost,0.000800,-0.017000,-0.017800
HistGradientBoosting,0.002600,0.001500,-0.001100
Logistic Regression,-0.000400,0.048400,0.048800


In [7]:
# Table 2(연도별 표본/클래스 비율)
bal = pd.read_csv(OUT_DIR / "year_asymmetry_class_balance.csv")
table2 = pd.DataFrame({
    "Year": bal["year"],
    "n": bal["n"].map("{:,}".format),
    "cost_barrier prevalence": (bal["prevalence"]*100).round(1).astype(str)+"%",
})


table2.style.hide(axis="index")

Year,n,cost_barrier prevalence
2022,"4,210",47.7%
2023,"3,760",48.6%


In [9]:
# table 3 (변수별 분표 비교)

var = pd.read_csv(OUT_DIR / "year_asymmetry_variable_tests.csv")
var = var[var["variable"] != "cost_barrier"]

table3 = pd.DataFrame({
    "Variable": var["variable"],
    "Chi-square p": var["p_value"].map(lambda p: f"{p:.4f}"),
    "Cramér's V": var["cramers_v"].map(lambda v: f"{v:.4f}"),
    "Max category diff (pp)": (var["max_category_prop_diff"]*100).round(1),
})

table3.style.hide(axis="index")

Variable,Chi-square p,Cramér's V,Max category diff (pp)
AGE3,0.9919,0.0175,0.400000
NEWRACE2,0.0015,0.0519,2.500000
IRINSUR4,0.0318,0.0240,1.400000
substance_peer_support,0.1493,0.0162,0.200000
mental_health_peer_support,0.7404,0.0037,0.100000


In [11]:
# table 4 (Adversarial validation)

adv = pd.read_csv(OUT_DIR / "year_asymmetry_adversarial_validation.csv")

table4= pd.DataFrame({
    "Model": adv["model"],
    "Year-classification AUC (5-fold CV)": [
        f"{m:.3f} ± {s:.3f}" for m, s
        in zip(adv["adv_auc_mean"], adv["adv_auc_std"])
    ],
})

table4.style.hide(axis="index")

Model,Year-classification AUC (5-fold CV)
Logistic Regression,0.514 ± 0.014
Random Forest,0.505 ± 0.009


In [ ]:
# another experiment - downsampling the majority class to match the minority class size, and then training the model on this balanced dataset. 
# This is done to see if balancing the classes improves the model's performance.

